# Yeni Araç Verisi — Temizleme Notebook'u
**Kaynak:** `yeni_arabalar_veriseti_part4.xlsx`  
**Hedef:** Mevcut pipeline ile uyumlu `verisetiTemiz2.xlsx` formatında çıktı üretmek  
**Çıktı:** `yeni_arabalar_temiz.xlsx`

## 1. Kütüphaneler ve Veri Yükleme

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('yeni_arabalar_veriseti.xlsx')
print(f'Ham veri: {df.shape[0]} satır, {df.shape[1]} sütun')
df.head(3)

Ham veri: 133 satır, 32 sütun


,url,baslik,konum,fiyat,resim_url,resim_klasor_yolu,ilan_tarihi,marka,seri,model,...,kimden,tramer,boya_degisen_detay,satici_adi,satici_tipi,yetkili_kisi,yetki_belge_no,aciklama,ortalama_yakit_tuketimi,yakit_deposu
0,https://www.arabam.com/ilan/galeriden-satilik-...,DEĞİŞENSİZ EKONOMİK AİLE ARACI,"Menderes Mh. Dulkadiroğlu, Kahramanmaraş",1.090.000 TL,https://arbstorage.mncdn.com/ilanfotograflari/...,C:\Users\Yasin\Desktop\Ders\web kazıma\indiril...,26 Mayıs 2026,Hyundai,i20,1.4 MPI Jump,...,Galeriden,Tramer\n9200,Sağ Arka Çamurluk:Orjinal | Arka Kaput:Orjinal...,CİHAN OTOMOTİV 046,Platinum Üye 3. YIL,Yunus Solak,4600072,Aracımız 2021 Çıkışlı Servis Bakımlı Herhangi ...,NaN,NaN
1,https://www.arabam.com/ilan/galeriden-satilik-...,ZAFERDEN 2017 BMW 118İ JOY PLUS 93.000KM DE SU...,"Davraz Mh. Merkez, Isparta",1.415.000 TL,https://arbstorage.mncdn.com/ilanfotograflari/...,C:\Users\Yasin\Desktop\Ders\web kazıma\indiril...,26 Mayıs 2026,BMW,1 Serisi,118i Joy Plus,...,Galeriden,Belirtilmemiş,Arka Kaput:Orjinal | Sol Arka Çamurluk:Orjinal...,ZAFER YAVUZ OTOMOTİV,Platinum Üye 9. YIL,Zafer YAVUZ,3200036,ARACIMIZ MÜKEMMEL TEMİZLİKTEDİR\nİLK SAHİBİNDE...,"5,2 lt",52 lt
2,https://www.arabam.com/ilan/galeriden-satilik-...,Erkut Yıldırım Auto/E 180 AMG/Cam Tvn/Keyless-...,"Ata Mh. Efeler, Aydın",3.050.000 TL,https://arbstorage.mncdn.com/ilanfotograflari/...,C:\Users\Yasin\Desktop\Ders\web kazıma\indiril...,25 Mayıs 2026,Mercedes - Benz,E,180 AMG,...,Galeriden,Belirtilmemiş,Sağ Arka Çamurluk:Orjinal | Arka Kaput:Orjinal...,ERKUT YILDIRIM AUTO,Platinum Üye 6. YIL,Erkut Yıldırım,0900154,2018 Model Mercedes Benz E 180 AMG 9G-Tronic 1...,"6,5 lt",50 lt


In [3]:
# Sütun isimleri ve ilk null özeti
print('Sütunlar:', list(df.columns))
print()
print('Null Sayıları:')
df.isnull().sum().sort_values(ascending=False)

Sütunlar: ['url', 'baslik', 'konum', 'fiyat', 'resim_url', 'resim_klasor_yolu', 'ilan_tarihi', 'marka', 'seri', 'model', 'yil', 'kilometre', 'vites_tipi', 'yakit_tipi', 'kasa_tipi', 'renk', 'motor_hacmi', 'motor_gucu', 'cekis', 'arac_durumu', 'boya_degisen', 'takasa_uygun', 'kimden', 'tramer', 'boya_degisen_detay', 'satici_adi', 'satici_tipi', 'yetkili_kisi', 'yetki_belge_no', 'aciklama', 'ortalama_yakit_tuketimi', 'yakit_deposu']

Null Sayıları:


ortalama_yakit_tuketimi    44
yakit_deposu               40
takasa_uygun                4
baslik                      0
aciklama                    0
yetki_belge_no              0
yetkili_kisi                0
satici_tipi                 0
satici_adi                  0
boya_degisen_detay          0
tramer                      0
kimden                      0
boya_degisen                0
arac_durumu                 0
cekis                       0
motor_gucu                  0
url                         0
renk                        0
kasa_tipi                   0
yakit_tipi                  0
vites_tipi                  0
kilometre                   0
yil                         0
model                       0
seri                        0
marka                       0
ilan_tarihi                 0
resim_klasor_yolu           0
resim_url                   0
fiyat                       0
konum                       0
motor_hacmi                 0
dtype: int64

## 2. Gereksiz Sütunları Düşür

In [6]:
# Model ile kullanılmayan ve kişisel/sistem sütunları
drop_cols = [
    'url', 'resim_url', 'resim_klasor_yolu',   # sistem
    'satici_adi', 'satici_tipi',               # satıcı profili
    'yetkili_kisi', 'yetki_belge_no',          # yetki bilgisi
    'aciklama',                                 # serbest metin
    'boya_degisen_detay',                       # parça detayı — boya_degisen yeterli
    'ortalama_yakit_tuketimi', 'yakit_deposu', # büyük oranda boş
    'takasa_uygun',                             # model özelliği değil
    'arac_durumu',                              # tüm satırlar 'İkinci El'
]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
print(f'Kalan sütunlar ({len(df.columns)}): {list(df.columns)}')

Kalan sütunlar (19): ['baslik', 'konum', 'fiyat', 'ilan_tarihi', 'marka', 'seri', 'model', 'yil', 'kilometre', 'vites_tipi', 'yakit_tipi', 'kasa_tipi', 'renk', 'motor_hacmi', 'motor_gucu', 'cekis', 'boya_degisen', 'kimden', 'tramer']


## 3. Fiyat Temizleme
Format: `"1.090.000 TL"` — bazılarında iki fiyat var: `"625.000 TL\n595.000 TL"` (indirimli fiyat)

In [9]:
def parse_fiyat(x):
    """
    '1.090.000 TL'       → 1090000
    '625.000 TL\n595.000 TL' → 595000  (indirimli = düşük fiyat)
    """
    x = str(x)
    # İki fiyat varsa en düşüğünü al (indirimi yansıtır)
    if '\n' in x:
        parcalar = x.split('\n')
        sayilar = []
        for p in parcalar:
            temiz = re.sub(r'[^\d]', '', p)
            if temiz:
                sayilar.append(int(temiz))
        return min(sayilar) if sayilar else np.nan
    else:
        temiz = re.sub(r'[^\d]', '', x)
        return int(temiz) if temiz else np.nan

df['fiyat'] = df['fiyat'].apply(parse_fiyat)
print(df['fiyat'].describe())

count    1.330000e+02
mean     1.103309e+06
std      8.981734e+05
min      1.100000e+05
25%      6.550000e+05
50%      9.200000e+05
75%      1.345000e+06
max      8.675000e+06
Name: fiyat, dtype: float64


In [11]:
# Makul fiyat aralığı filtresi (eğitim verisiyle aynı sınır)
once = len(df)
df = df[(df['fiyat'] > 50000) & (df['fiyat'] < 3000000)]
print(f'Fiyat filtresi: {once} → {len(df)} satır ({once - len(df)} düşürüldü)')

Fiyat filtresi: 133 → 129 satır (4 düşürüldü)


## 4. Kilometre Temizleme

In [14]:
# '127.000 km' → 127000
df = df[df['kilometre'] != '']
df['kilometre'] = (
    df['kilometre']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(' km', '', regex=False)
    .str.strip()
)
df['kilometre'] = pd.to_numeric(df['kilometre'], errors='coerce')

# Makul aralık
df = df[(df['kilometre'] >= 0) & (df['kilometre'] <= 500000)]
print(df['kilometre'].describe())

count       129.000000
mean     154424.271318
std       90553.709702
min        6001.000000
25%       93000.000000
50%      140000.000000
75%      218000.000000
max      365000.000000
Name: kilometre, dtype: float64


## 5. Motor Hacmi Temizleme
Formatlar: `"1201 - 1400 cm3"`, `"1499 cc"`, `"1595 cc"`, `"-"` (elektrikli)

In [17]:
def temizle_motor_hacmi(x):
    if pd.isnull(x):
        return np.nan
    x = str(x).lower().replace('cc', '').replace('cm3', '').strip()
    # Elektrikli araçlar: '-' veya boş
    if x in ['-', '', 'nan']:
        return np.nan
    # Aralık formatı: '1201 - 1400'
    if '-' in x:
        parcalar = [p.strip() for p in x.split('-') if p.strip()]
        try:
            return (float(parcalar[0]) + float(parcalar[-1])) / 2
        except:
            return np.nan
    else:
        try:
            return float(x)
        except:
            return np.nan

df['motor_hacmi'] = df['motor_hacmi'].apply(temizle_motor_hacmi)

# Başlıktan hacim çıkar (eksik olanlar için)
def extract_engine_from_baslik(text):
    if pd.isnull(text):
        return None
    match = re.search(r'(\d\.\d)', str(text))
    if match:
        return float(match.group(1)) * 1000
    return None

if 'baslik' in df.columns:
    df['motor_hacmi'] = df.apply(
        lambda row: extract_engine_from_baslik(row['baslik'])
        if pd.isnull(row['motor_hacmi']) else row['motor_hacmi'],
        axis=1
    )

# Marka bazlı medyan ile doldur (elektrikli dahil)
df['motor_hacmi'] = df.groupby('marka')['motor_hacmi'].transform(
    lambda x: x.fillna(x.median())
)
df['motor_hacmi'] = df['motor_hacmi'].fillna(df['motor_hacmi'].median())

# 100cc yuvarlama (eğitim pipeline ile aynı)
df['motor_hacmi'] = (df['motor_hacmi'] / 100).round() * 100

# Makul aralık filtresi
df = df[(df['motor_hacmi'] >= 800) & (df['motor_hacmi'] <= 3000)]
print(df['motor_hacmi'].describe())

count     128.000000
mean     1417.187500
std       214.479548
min       900.000000
25%      1300.000000
50%      1500.000000
75%      1500.000000
max      2500.000000
Name: motor_hacmi, dtype: float64


## 6. Motor Gücü Temizleme
Formatlar: `"76 - 100 HP"`, `"136 hp"`, `"156 hp"`

In [20]:
def temizle_motor_gucu(x):
    if pd.isnull(x):
        return np.nan
    x = str(x).lower().replace('hp', '').strip()
    if x in ['-', '', 'nan']:
        return np.nan
    if '-' in x:
        parcalar = [p.strip() for p in x.split('-') if p.strip()]
        try:
            return (float(parcalar[0]) + float(parcalar[-1])) / 2
        except:
            return np.nan
    else:
        try:
            return float(x)
        except:
            return np.nan

df['motor_gucu'] = df['motor_gucu'].apply(temizle_motor_gucu)

# Makul aralık
df = df[(df['motor_gucu'] >= 50) & (df['motor_gucu'] <= 400)]

# Marka bazlı medyan dolgu
df['motor_gucu'] = df.groupby('marka')['motor_gucu'].transform(
    lambda x: x.fillna(x.median())
)
df['motor_gucu'] = df['motor_gucu'].fillna(df['motor_gucu'].median())
print(df['motor_gucu'].describe())

count    128.000000
mean     106.906250
std       24.595127
min       65.000000
25%       88.000000
50%      100.000000
75%      120.250000
max      188.000000
Name: motor_gucu, dtype: float64


## 7. Tramer Temizleme
Formatlar: `"Tramer\n9200"`, `"Belirtilmemiş"`, `"yok"`, `"Bu araç ağır hasar kayıdır"`

In [23]:
def parse_tramer(x):
    if pd.isnull(x):
        return np.nan
    x = str(x).strip()
    # Belirtilmemiş / yok
    lower = x.lower()
    if lower in ['belirtilmemiş', 'yok', 'nan', '']:
        return 0
    # Ağır hasar: çok yüksek bir sabit değer ata (999999)
    if 'ağır hasar' in lower or 'agir hasar' in lower:
        return 999999
    # 'Tramer\n9200' formatı
    if '\n' in x:
        sayi = x.split('\n')[-1].strip()
        try:
            return float(sayi)
        except:
            return np.nan
    # Sayısal
    temiz = re.sub(r'[^\d.]', '', x)
    try:
        return float(temiz)
    except:
        return 0

df['tramer'] = df['tramer'].apply(parse_tramer)

# Ağır hasar araçları filtrele (999999 olarak işaretlenenler)
print(f'Ağır hasar araç sayısı: {(df["tramer"] == 999999).sum()}')
df = df[df['tramer'] != 999999]

# tramer > 500000 olanları filtrele
df = df[df['tramer'] <= 500000]
df['tramer'] = df['tramer'].fillna(0)
print(df['tramer'].describe())

Ağır hasar araç sayısı: 2
count       126.000000
mean       4437.785714
std       16537.442046
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max      130000.000000
Name: tramer, dtype: float64


In [25]:
# tramer_bilinmiyor bayrağı (eski notebook'ta hasar_bilinmiyor → tramer_bilinmiyor)
# Burada tramer 0 olanlar ya 'yok' ya 'belirtilmemiş' — ikisini ayırt edemiyoruz
# Temiz veriyle uyum için: parse sırasında NaN dönenler bilinmiyor
df['tramer_bilinmiyor'] = 0   # parse_tramer zaten NaN'ı 0'a çevirdi

def tramer_kategori(x):
    if pd.isna(x) or x == 0:
        return 'yok'
    elif x < 5000:
        return 'dusuk'
    elif x < 20000:
        return 'orta'
    else:
        return 'yuksek'

df['tramer_kategori'] = df['tramer'].apply(tramer_kategori)
print(df['tramer_kategori'].value_counts())

tramer_kategori
yok       106
yuksek      9
orta        7
dusuk       4
Name: count, dtype: int64


## 8. Boya/Değişen Temizleme
Formatlar: `"2 boyalı, 3 lokal boyalı"`, `"1 değişen, 1 boyalı"`, `"Tamamı orjinal"`, `"Tamamı boyalı"`

In [28]:
def parse_boya_degisen(x):
    """
    Döndürür: (boyali_sayisi, degisen_sayisi)
    
    Lokal boyalı → boyalı sayısına eklenir (daha az hasarlı ama yine de boyalı)
    Tamamı orjinal → (0, 0)
    Tamamı boyalı → sabit 11 boyalı varsayımı
    Belirtilmemiş → (NaN, NaN)
    """
    if pd.isnull(x):
        return np.nan, np.nan
    x_str = str(x).lower()

    # Tamamı orijinal
    if 'orjinal' in x_str or 'orijinal' in x_str:
        return 0, 0

    # Belirtilmemiş
    if 'belirtilmemiş' in x_str:
        return np.nan, np.nan

    # Tamamı boyalı (sayı yok)
    if 'tamamı boyalı' in x_str and not re.search(r'\d', x_str):
        return 11, 0

    boyali = 0
    degisen = 0

    # Değişen sayısı
    d = re.search(r'(\d+)\s*değişen', x_str)
    if d:
        degisen = int(d.group(1))

    # Normal boyalı
    b = re.search(r'(\d+)\s*boyalı', x_str)
    if b:
        # 'lokal boyalı' öncesinde gelen rakamı yakalamadığımızdan emin ol
        match_str = x_str[:b.start(1) + len(b.group(1))]
        if 'lokal' not in match_str[-15:]:
            boyali += int(b.group(1))

    # Lokal boyalı → boyalı sayısına ekle
    lk = re.search(r'(\d+)\s*lokal\s*boyalı', x_str)
    if lk:
        boyali += int(lk.group(1))

    return boyali, degisen

df[['boyali_sayisi', 'degisen_sayisi']] = df['boya_degisen'].apply(
    lambda x: pd.Series(parse_boya_degisen(x))
)

# hasar_bilinmiyor → tramer_bilinmiyor (eski notebook adlandırması)
df['tramer_bilinmiyor'] = df['boyali_sayisi'].isnull().astype(int)
df['boyali_sayisi'] = df['boyali_sayisi'].fillna(0)
df['degisen_sayisi'] = df['degisen_sayisi'].fillna(0)

print('Boyalı sayısı dağılımı:')
print(df['boyali_sayisi'].value_counts().sort_index())
print('\nDeğişen sayısı dağılımı:')
print(df['degisen_sayisi'].value_counts().sort_index())

Boyalı sayısı dağılımı:
boyali_sayisi
0.0     51
1.0     23
2.0      9
3.0     18
4.0      6
5.0      8
6.0      2
8.0      2
9.0      1
10.0     3
11.0     3
Name: count, dtype: int64

Değişen sayısı dağılımı:
degisen_sayisi
0.0    96
1.0    17
2.0     8
3.0     4
5.0     1
Name: count, dtype: int64


## 9. Tarih Temizleme
Format: `"26 Mayıs 2026"` — Türkçe metin tarih

In [31]:
AY_MAP = {
    'ocak': 1, 'şubat': 2, 'mart': 3, 'nisan': 4,
    'mayıs': 5, 'haziran': 6, 'temmuz': 7, 'ağustos': 8,
    'eylül': 9, 'ekim': 10, 'kasım': 11, 'aralık': 12
}

def parse_turkce_tarih(x):
    if pd.isnull(x):
        return pd.NaT
    parcalar = str(x).strip().lower().split()
    if len(parcalar) == 3:
        try:
            gun = int(parcalar[0])
            ay  = AY_MAP.get(parcalar[1], 0)
            yil = int(parcalar[2])
            if ay:
                return pd.Timestamp(year=yil, month=ay, day=gun)
        except:
            pass
    return pd.NaT

df['ilan_tarihi'] = df['ilan_tarihi'].apply(parse_turkce_tarih)
df['ilan_yil'] = df['ilan_tarihi'].dt.year
df['ilan_ay']  = df['ilan_tarihi'].dt.month
df.drop(columns=['ilan_tarihi'], inplace=True)

print('İlan yılı dağılımı:')
print(df['ilan_yil'].value_counts())

İlan yılı dağılımı:
ilan_yil
2026    126
Name: count, dtype: int64


## 10. Konum → Şehir

In [34]:
# 'Menderes Mh. Dulkadiroğlu, Kahramanmaraş' → 'Kahramanmaraş'
df['sehir'] = df['konum'].apply(
    lambda x: str(x).split(',')[-1].strip() if pd.notna(x) else np.nan
)
df.drop(columns=['konum'], inplace=True)
print(df['sehir'].value_counts().head(15))

sehir
İstanbul     19
Ankara       12
Konya        10
İzmir         9
Aydın         8
Gaziantep     7
Antalya       6
Samsun        6
Kayseri       5
Adana         3
Eskişehir     3
Yozgat        3
Bursa         3
Bolu          3
Hatay         2
Name: count, dtype: int64


## 11. Marka + Seri Birleştirme

In [37]:
# Eski notebook: df['marka'] = df['marka'] + '_' + df['seri']
df['marka'] = df['marka'].astype(str) + '_' + df['seri'].astype(str)
df.drop(columns=['seri'], inplace=True)
print('Marka örnekleri:')
print(df['marka'].value_counts().head(10))

Marka örnekleri:
marka
Fiat_Egea          12
Renault_Megane      9
Renault_Clio        9
Toyota_Corolla      8
Opel_Astra          7
Renault_Fluence     6
Skoda_Octavia       6
Honda_Civic         5
Ford_Focus          5
Hyundai_i20         5
Name: count, dtype: int64


## 12. Renk Temizleme

In [40]:
df.dropna(subset=['renk'], inplace=True)
# Parantez içindeki açıklamaları kaldır: 'Gri (Füme)' → 'Gri'
df['renk'] = df['renk'].apply(lambda x: re.sub(r'\s*\(.*\)', '', str(x)).strip())

# Nadir renkleri 'Diger' yap (eşik: 200 — orijinal notebook ile aynı)
# Yeni veri az olduğundan eşiği 5'e düşürüyoruz
renk_counts = df['renk'].value_counts()
rare_colors = renk_counts[renk_counts < 5].index
df['renk'] = df['renk'].replace(rare_colors, 'Diger')
print(df['renk'].value_counts())

renk
Beyaz      53
Gri        28
Diger      12
Siyah      11
Mavi        9
Kırmızı     7
Füme        6
Name: count, dtype: int64


## 13. Çekiş Temizleme

In [43]:
df['cekis'] = df['cekis'].replace('-', pd.NA)
df['cekis'] = df['cekis'].fillna(df['cekis'].mode()[0])
print(df['cekis'].value_counts())

cekis
Önden Çekiş     119
Arkadan İtiş      7
Name: count, dtype: int64


## 14. Yaş Hesaplama

In [46]:
df['yas'] = 2026 - df['yil']  # Yeni veri 2026 yılına ait
df.drop(columns=['yil'], inplace=True)
print(df['yas'].describe())

count    126.000000
mean      11.222222
std        7.328726
min        0.000000
25%        5.250000
50%       11.000000
75%       14.000000
max       32.000000
Name: yas, dtype: float64


## 15. Baslik Düşür

In [49]:
if 'baslik' in df.columns:
    df.drop(columns=['baslik'], inplace=True)
df.drop(columns=['boya_degisen'], inplace=True, errors='ignore')
print(f'Temizlenen sütunlar: {list(df.columns)}')

Temizlenen sütunlar: ['fiyat', 'marka', 'model', 'kilometre', 'vites_tipi', 'yakit_tipi', 'kasa_tipi', 'renk', 'motor_hacmi', 'motor_gucu', 'cekis', 'kimden', 'tramer', 'tramer_bilinmiyor', 'tramer_kategori', 'boyali_sayisi', 'degisen_sayisi', 'ilan_yil', 'ilan_ay', 'sehir', 'yas']


## 16. Model Sütunu Temizleme

In [52]:
df.dropna(subset=['model'], inplace=True)
df = df[df['model'] != '']

# Çok nadir modelleri 'diger' yap
model_counts = df['model'].value_counts()
rare_models = model_counts[model_counts < 2].index
df['model'] = df['model'].apply(lambda x: 'diger' if x in rare_models else x)
print(f'Model çeşidi: {df["model"].nunique()}')

Model çeşidi: 19


## 17. kimden Null Kontrolü

In [55]:
df['kimden'] = df['kimden'].fillna('Sahibinden')
print(df['kimden'].value_counts())

kimden
Galeriden     122
Sahibinden      4
Name: count, dtype: int64


## 18. Final Null Kontrolü ve Sütun Sıralaması

In [58]:
print('Null Özeti:')
print(df.isnull().sum())
print(f'\nFinal veri boyutu: {df.shape}')

Null Özeti:
fiyat                0
marka                0
model                0
kilometre            0
vites_tipi           0
yakit_tipi           0
kasa_tipi            0
renk                 0
motor_hacmi          0
motor_gucu           0
cekis                0
kimden               0
tramer               0
tramer_bilinmiyor    0
tramer_kategori      0
boyali_sayisi        0
degisen_sayisi       0
ilan_yil             0
ilan_ay              0
sehir                0
yas                  0
dtype: int64

Final veri boyutu: (126, 21)


In [60]:
# Temiz verisetindeki sütun sıralamasına getir
HEDEF_SUTUNLAR = [
    'marka', 'model', 'kilometre', 'vites_tipi', 'yakit_tipi', 'kasa_tipi',
    'renk', 'motor_hacmi', 'motor_gucu', 'cekis', 'kimden', 'tramer',
    'boyali_sayisi', 'degisen_sayisi', 'tramer_bilinmiyor', 'tramer_kategori',
    'sehir', 'ilan_yil', 'ilan_ay', 'yas', 'fiyat'
]

# Eksik sütunları kontrol et
eksik = [s for s in HEDEF_SUTUNLAR if s not in df.columns]
print('Eksik sütunlar:', eksik)

df_final = df[HEDEF_SUTUNLAR].copy()
df_final = df_final.reset_index(drop=True)
print(f'\nFinal DataFrame: {df_final.shape}')
df_final.head()

Eksik sütunlar: []

Final DataFrame: (126, 21)


,marka,model,kilometre,vites_tipi,yakit_tipi,kasa_tipi,renk,motor_hacmi,motor_gucu,cekis,...,tramer,boyali_sayisi,degisen_sayisi,tramer_bilinmiyor,tramer_kategori,sehir,ilan_yil,ilan_ay,yas,fiyat
0,Hyundai_i20,diger,127000,Otomatik,Benzin,Hatchback/5,Beyaz,1300.0,88.0,Önden Çekiş,...,9200.0,5.0,0.0,0,orta,Kahramanmaraş,2026,5,6,1090000
1,BMW_1 Serisi,diger,93000,Otomatik,Benzin,Hatchback/5,Beyaz,1500.0,136.0,Arkadan İtiş,...,0.0,1.0,1.0,0,yok,Isparta,2026,5,9,1415000
2,Toyota_Corolla,1.33 Life,180000,Düz,LPG & Benzin,Sedan,Füme,1300.0,88.0,Önden Çekiş,...,0.0,1.0,0.0,0,yok,Kütahya,2026,5,10,925000
3,Volkswagen_Passat,1.6 TDi BlueMotion Comfortline,93000,Düz,Dizel,Sedan,Gri,1600.0,120.0,Önden Çekiş,...,0.0,0.0,0.0,0,yok,Afyonkarahisar,2026,5,9,1719750
4,Volkswagen_Polo,diger,180000,Yarı Otomatik,Dizel,Hatchback/5,Gri,1400.0,90.0,Önden Çekiş,...,0.0,4.0,0.0,0,yok,Ankara,2026,5,9,915000


In [62]:
df_final.describe()

,kilometre,motor_hacmi,motor_gucu,tramer,boyali_sayisi,degisen_sayisi,tramer_bilinmiyor,ilan_yil,ilan_ay,yas,fiyat
count,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000,126.0,126.0,126.000000,1.260000e+02
mean,155008.420635,1415.079365,106.793651,4437.785714,2.055556,0.396825,0.047619,2026.0,5.0,11.222222,9.907949e+05
std,91489.126636,215.431645,24.774283,16537.442046,2.725599,0.849276,0.213809,0.0,0.0,7.328726,5.174665e+05
min,6001.000000,900.000000,65.000000,0.000000,0.000000,0.000000,0.000000,2026.0,5.0,0.000000,1.100000e+05
25%,93000.000000,1300.000000,88.000000,0.000000,0.000000,0.000000,0.000000,2026.0,5.0,5.250000,6.375000e+05
50%,141000.000000,1500.000000,100.000000,0.000000,1.000000,0.000000,0.000000,2026.0,5.0,11.000000,8.770000e+05
75%,224000.000000,1500.000000,120.750000,0.000000,3.000000,0.000000,0.000000,2026.0,5.0,14.000000,1.317499e+06
max,365000.000000,2500.000000,188.000000,130000.000000,11.000000,5.000000,1.000000,2026.0,5.0,32.000000,2.975000e+06


## 19. Excel'e Kaydet

In [64]:
output_path = 'yeni_arabalar_temiz126.xlsx'
df_final.to_excel(output_path, index=False)
print(f'✅ Kaydedildi: {output_path}')
print(f'   Satır: {len(df_final)}, Sütun: {len(df_final.columns)}')
print(f'   Sütunlar: {list(df_final.columns)}')

✅ Kaydedildi: yeni_arabalar_temiz126.xlsx
   Satır: 126, Sütun: 21
   Sütunlar: ['marka', 'model', 'kilometre', 'vites_tipi', 'yakit_tipi', 'kasa_tipi', 'renk', 'motor_hacmi', 'motor_gucu', 'cekis', 'kimden', 'tramer', 'boyali_sayisi', 'degisen_sayisi', 'tramer_bilinmiyor', 'tramer_kategori', 'sehir', 'ilan_yil', 'ilan_ay', 'yas', 'fiyat']
